# Homework 4 — Filters as Features

**Student:** Daniela Gonzalez  
**Course:** Computer Vision and Speech Recognition  
**Dataset:** MILK10k

In [1]:
# --- Setup ---
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
from scipy import ndimage
import cv2

ROOT = Path.cwd().resolve()
if (ROOT / "notebooks").is_dir():
    sys.path.insert(0, str(ROOT))
elif (ROOT.parent / "notebooks").is_dir():
    sys.path.insert(0, str(ROOT.parent))

from milk10k_pipeline import config

FIG = config.REPORTS_DIR / "figures_hw4"
FIG.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(config.METADATA_PATH)
print("metadata:", df.shape)
print("classes:", df["diagnosis_1"].value_counts().to_dict())

metadata: (10480, 17)
classes: {'Malignant': 7268, 'Benign': 2966, 'Indeterminate': 246}


## Exercise 1 — Explore learned filters

Apply a set of filters (blur, Sobel, Laplacian, Gabor) to one image
from each diagnosis class and compare visually whether edges or
textures look different between malignant and benign lesions.


In [2]:
# --- Exercise 1: filter bank on one image per class -------------------------
sample_ids = {}
for cls in ["Benign", "Malignant", "Indeterminate"]:
    row = df[df["diagnosis_1"] == cls].sort_values("isic_id").iloc[0]
    sample_ids[cls] = row["isic_id"]
print("sample ids:", sample_ids)


def load_gray(isic_id):
    """Load a MILK10k image and convert to float grayscale in [0, 1]."""
    path = config.image_path(isic_id)
    img = Image.open(path).convert("L")
    return np.asarray(img, dtype=np.float32) / 255.0


def gabor_kernel(ksize=31, sigma=4.0, theta=0.0, lambd=10.0, gamma=0.5):
    """Build a Gabor kernel with cv2.getGaborKernel."""
    return cv2.getGaborKernel((ksize, ksize), sigma, theta, lambd, gamma, psi=0)


def apply_filter_bank(gray):
    """Return a dict of filtered images."""
    out = {}
    out["original"] = gray
    out["gaussian_blur"] = ndimage.gaussian_filter(gray, sigma=2.0)
    out["sobel_x"] = np.abs(ndimage.sobel(gray, axis=1))
    out["sobel_y"] = np.abs(ndimage.sobel(gray, axis=0))
    out["laplacian"] = np.abs(ndimage.laplace(gray))
    k_v = gabor_kernel(theta=0)
    k_h = gabor_kernel(theta=np.pi / 2)
    out["gabor_0deg"] = np.abs(cv2.filter2D(gray, cv2.CV_32F, k_v))
    out["gabor_90deg"] = np.abs(cv2.filter2D(gray, cv2.CV_32F, k_h))
    return out


filter_names = ["original", "gaussian_blur", "sobel_x", "sobel_y",
                "laplacian", "gabor_0deg", "gabor_90deg"]
classes = ["Benign", "Malignant", "Indeterminate"]

fig, axes = plt.subplots(len(classes), len(filter_names),
                         figsize=(3 * len(filter_names), 3 * len(classes)))
for r, cls in enumerate(classes):
    isic = sample_ids[cls]
    gray = load_gray(isic)
    filtered = apply_filter_bank(gray)
    for c, name in enumerate(filter_names):
        axes[r, c].imshow(filtered[name], cmap="gray")
        axes[r, c].axis("off")
        if r == 0:
            axes[r, c].set_title(name, fontsize=9)
    axes[r, 0].set_ylabel(cls, fontsize=10)
plt.tight_layout()
out_path = FIG / "ex1_filter_bank_per_class.png"
fig.savefig(out_path, dpi=120)
plt.close(fig)
print("saved:", out_path)

rows = []
for cls in classes:
    isic = sample_ids[cls]
    gray = load_gray(isic)
    filtered = apply_filter_bank(gray)
    for name, img in filtered.items():
        rows.append({
            "class": cls,
            "filter": name,
            "mean": float(img.mean()),
            "std": float(img.std()),
            "p95": float(np.percentile(img, 95)),
        })
summary = pd.DataFrame(rows)
print()
print("Filter summary per class:")
print(summary.round(4).to_string(index=False))
summary.to_csv(FIG / "ex1_filter_summary.csv", index=False)
print("saved:", FIG / "ex1_filter_summary.csv")


sample ids: {'Benign': 'ISIC_0073863', 'Malignant': 'ISIC_0051817', 'Indeterminate': 'ISIC_0141634'}


saved: /Users/danielagonzalez/Downloads/milk10k-eda/reports/figures_hw4/ex1_filter_bank_per_class.png

Filter summary per class:
        class        filter   mean    std    p95
       Benign      original 0.5008 0.1667 0.6353
       Benign gaussian_blur 0.5008 0.1658 0.6311
       Benign       sobel_x 0.0373 0.0543 0.1020
       Benign       sobel_y 0.0439 0.0537 0.1176
       Benign     laplacian 0.0124 0.0191 0.0392
       Benign    gabor_0deg 4.0692 1.3865 5.2781
       Benign   gabor_90deg 4.0605 1.4123 5.4668
    Malignant      original 0.4234 0.1185 0.6196
    Malignant gaussian_blur 0.4234 0.1115 0.6070
    Malignant       sobel_x 0.1053 0.1173 0.3333
    Malignant       sobel_y 0.1046 0.1179 0.3255
    Malignant     laplacian 0.0281 0.0358 0.0941
    Malignant    gabor_0deg 3.4785 1.3777 5.6756
    Malignant   gabor_90deg 3.4591 1.3837 5.7346
Indeterminate      original 0.5782 0.0962 0.7412
Indeterminate gaussian_blur 0.5782 0.0937 0.7370
Indeterminate       sobel_x 0.0442 0.0

### Exercise 1 — Written answer

**What each filter shows.** Gaussian blur removes fine dermoscopic
texture (hairs, pigment dots) and keeps the overall lesion shape.
Sobel x and Sobel y respond to vertical and horizontal borders: hairs
appear as thin bright lines, the lesion border as a thicker ring.
Laplacian responds to any local difference, so it highlights spots and
hairs. The two Gabor filters highlight stripes running in their own
direction.

**Do Malignant and Benign look different? Yes, and the quantitative
summary confirms it.** The most informative filters are the edge
detectors:

| filter | Benign (mean) | Malignant (mean) | ratio |
|---|---|---|---|
| sobel_x | 0.0373 | 0.1053 | **2.8x** |
| sobel_y | 0.0439 | 0.1046 | **2.4x** |
| laplacian | 0.0124 | 0.0281 | **2.3x** |
| gaussian_blur | 0.5008 | 0.4234 | 0.85x |
| gabor_0deg | 4.0692 | 3.4785 | 0.85x |

The malignant sample produces roughly **two to three times more edge
energy** than the benign one, both on Sobel and on Laplacian. This is
consistent with the clinical observation that malignant lesions tend
to have more irregular and higher-contrast borders. Gaussian blur and
Gabor are less discriminative in this single-image comparison: the
malignant value is actually lower, which shows that these filters are
capturing overall texture rather than border irregularity.

**But one image is not enough.** The comparison is on a single
image per class, so the difference could be due to the specific lesion
chosen rather than to the class. The visual grid in
`ex1_filter_bank_per_class.png` shows the pattern clearly, but to
claim that edge energy is a reliable discriminative feature we would
need to compute the same summary over many images per class and test
the difference statistically. That is what a CNN does automatically:
it learns which filters (and which combinations) best separate the
classes, instead of us guessing.


## Exercise 2 — Gaussian blur: what does sigma control?

Read the documentation of `scipy.ndimage.gaussian_filter`, identify what
the `sigma` parameter controls, and experiment with different values.


In [3]:
# --- Exercise 2: gaussian_filter sigma sweep ---------------------------------
# Read the docstring to show what sigma does
import inspect
from scipy.ndimage import gaussian_filter
print("gaussian_filter signature:")
print(inspect.signature(gaussian_filter))
print()
print("First 20 lines of the docstring:")
print("\n".join(gaussian_filter.__doc__.splitlines()[:20]))

# Use one image (the malignant sample from Ex1)
sample_isic = "ISIC_0051817"
gray = load_gray(sample_isic)
print()
print("image shape:", gray.shape)

sigmas = [0.5, 1.0, 2.0, 4.0, 8.0, 16.0]
fig, axes = plt.subplots(1, len(sigmas) + 1,
                         figsize=(3 * (len(sigmas) + 1), 3))
axes[0].imshow(gray, cmap="gray")
axes[0].set_title("original", fontsize=9)
axes[0].axis("off")
for i, s in enumerate(sigmas):
    blurred = gaussian_filter(gray, sigma=s)
    axes[i + 1].imshow(blurred, cmap="gray")
    axes[i + 1].set_title(f"sigma={s}", fontsize=9)
    axes[i + 1].axis("off")
plt.tight_layout()
out_path = FIG / "ex2_gaussian_sigma_sweep.png"
fig.savefig(out_path, dpi=120)
plt.close(fig)
print("saved:", out_path)

# Quantitative: how does the standard deviation of the image change?
print()
print("Effect of sigma on image variance (smoothed image):")
for s in sigmas:
    blurred = gaussian_filter(gray, sigma=s)
    print(f"  sigma={s:5.1f}  std={blurred.std():.4f}  min={blurred.min():.4f}  max={blurred.max():.4f}")


gaussian_filter signature:
(input, sigma, order=0, output=None, mode='reflect', cval=0.0, truncate=4.0, *, radius=None, axes=None)

First 20 lines of the docstring:
Multidimensional Gaussian filter.

Parameters
----------
input : array_like
    The input array.
sigma : scalar or sequence of scalars
    Standard deviation for Gaussian kernel. The standard
    deviations of the Gaussian filter are given for each axis as a
    sequence, or as a single number, in which case it is equal for
    all axes.
order : int or sequence of ints, optional
    The order of the filter along each axis is given as a sequence
    of integers, or as a single number. An order of 0 corresponds
    to convolution with a Gaussian kernel. A positive order
    corresponds to convolution with that derivative of a Gaussian.
output : array or dtype, optional
    The array in which to place the output, or the dtype of the
    returned array. By default an array of the same dtype as input
    will be created.

image 

saved: /Users/danielagonzalez/Downloads/milk10k-eda/reports/figures_hw4/ex2_gaussian_sigma_sweep.png

Effect of sigma on image variance (smoothed image):
  sigma=  0.5  std=0.1175  min=0.0353  max=0.8036
  sigma=  1.0  std=0.1152  min=0.0555  max=0.7653
  sigma=  2.0  std=0.1115  min=0.0878  max=0.7236
  sigma=  4.0  std=0.1064  min=0.1168  max=0.6923
  sigma=  8.0  std=0.1003  min=0.1552  max=0.6675
  sigma= 16.0  std=0.0931  min=0.2056  max=0.6458


### Exercise 2 — Written answer

**What `sigma` controls.** The docstring of
`scipy.ndimage.gaussian_filter` says: *"Standard deviation for Gaussian
kernel. The standard deviations of the Gaussian filter are given for
each axis as a sequence, or as a single number, in which case it is
equal for all axes."*

So `sigma` is the **width of the Gaussian kernel in pixels**. It is
the only parameter that controls how much the image gets smoothed:

- `sigma = 0.5`: kernel is narrow, only nearby pixels contribute,
  fine texture is preserved.
- `sigma = 16`: kernel is wide, each output pixel is a weighted
  average of a large neighborhood, all fine structure disappears.

The related parameter `truncate = 4.0` controls how far from the
kernel center we cut the Gaussian (default: 4 sigma). Larger
`truncate` gives a more accurate Gaussian at the cost of more compute.

**What the sweep shows.** The 6-panel figure above
(`ex2_gaussian_sigma_sweep.png`) makes the effect visually obvious:
0.5 is essentially the original, 2 already removes hairs and small
structures, 16 leaves only a smooth blob where the lesion boundary is
still visible. The numeric table below confirms it:

| sigma | std | min | max |
|---|---|---|---|
| 0.5 | 0.1175 | 0.0353 | 0.8036 |
| 1.0 | 0.1152 | 0.0555 | 0.7653 |
| 2.0 | 0.1115 | 0.0878 | 0.7236 |
| 4.0 | 0.1064 | 0.1168 | 0.6923 |
| 8.0 | 0.1003 | 0.1552 | 0.6675 |
| 16.0 | 0.0931 | 0.2056 | 0.6458 |

As sigma grows, the standard deviation of the smoothed image
decreases (0.118 -> 0.093), the minimum pixel value rises (0.035 ->
0.206) and the maximum value falls (0.804 -> 0.646). In other words,
the pixel-value range **shrinks from both ends**: extreme dark and
extreme bright pixels get averaged out with their neighbors.

**Practical consequence for MILK10k.** The choice of sigma is a
trade-off. A small sigma keeps diagnostically meaningful textures
(pigment network, dots) but also keeps noise and hairs. A large sigma
removes noise and hair artefacts but may also erase the small
malignant patterns we care about. For this project, sigma in the
range 1 to 2 is a reasonable starting point: it removes sensor noise
and thin hairs without destroying the pigment network. This matches
what the first layer of a CNN trained on MILK10k would learn on its
own.


## Exercise 3 — Canny edge detection vs Sobel

Implement Canny edge detection from scratch (5 steps) and compare with
Sobel on one MILK10k image.


In [4]:
# --- Exercise 3: Canny from scratch vs Sobel ---------------------------------
def canny_from_scratch(gray, sigma=1.4, low_thresh=0.05, high_thresh=0.15):
    """
    Canny edge detector implemented from scratch.

    Steps
    -----
    1. Gaussian smoothing (noise reduction).
    2. Gradient computation (Sobel x, y).
    3. Gradient magnitude and direction.
    4. Non-maximum suppression along the gradient direction.
    5. Double threshold and hysteresis.

    Returns
    -------
    magnitude : np.ndarray, gradient magnitude
    edges : np.ndarray, binary edge map (0/1)
    """
    # 1. Gaussian smoothing
    smoothed = ndimage.gaussian_filter(gray, sigma=sigma)

    # 2. Sobel gradients
    gx = ndimage.sobel(smoothed, axis=1)
    gy = ndimage.sobel(smoothed, axis=0)

    # 3. Magnitude and direction
    magnitude = np.hypot(gx, gy)
    # Normalize to [0, 1] for stable thresholds
    if magnitude.max() > 0:
        magnitude = magnitude / magnitude.max()
    direction = np.arctan2(gy, gx)  # in radians

    # 4. Non-maximum suppression
    # Quantize direction to 4 bins: 0, 45, 90, 135 degrees
    angle = (np.rad2deg(direction) + 180) % 180
    nms = np.zeros_like(magnitude)
    H, W = magnitude.shape
    for i in range(1, H - 1):
        for j in range(1, W - 1):
            a = angle[i, j]
            if (a < 22.5) or (a >= 157.5):
                nb1, nb2 = magnitude[i, j - 1], magnitude[i, j + 1]
            elif (a < 67.5):
                nb1, nb2 = magnitude[i - 1, j + 1], magnitude[i + 1, j - 1]
            elif (a < 112.5):
                nb1, nb2 = magnitude[i - 1, j], magnitude[i + 1, j]
            else:
                nb1, nb2 = magnitude[i - 1, j - 1], magnitude[i + 1, j + 1]
            if magnitude[i, j] >= nb1 and magnitude[i, j] >= nb2:
                nms[i, j] = magnitude[i, j]

    # 5. Double threshold + hysteresis
    strong = nms >= high_thresh
    weak = (nms >= low_thresh) & (nms < high_thresh)
    # Connect weak pixels that touch strong ones (simple 8-connectivity pass)
    from scipy.ndimage import label
    # Label connected components of strong+weak, keep only those with a strong pixel
    combined = strong | weak
    lab, n_lab = label(combined, structure=np.ones((3, 3)))
    keep = np.zeros(n_lab + 1, dtype=bool)
    for k in range(1, n_lab + 1):
        if strong[lab == k].any():
            keep[k] = True
    edges = keep[lab].astype(np.float32)
    return magnitude, edges


# Run on the malignant sample
isic = "ISIC_0051817"
gray = load_gray(isic)
mag, canny_edges = canny_from_scratch(gray, sigma=1.4,
                                       low_thresh=0.05, high_thresh=0.15)

# Sobel for comparison
gx = ndimage.sobel(gray, axis=1)
gy = ndimage.sobel(gray, axis=0)
sobel_mag = np.hypot(gx, gy)
sobel_mag = sobel_mag / sobel_mag.max()
# Threshold Sobel at the same high threshold for a fair comparison
sobel_binary = (sobel_mag >= 0.15).astype(np.float32)

# Plot: original | Sobel magnitude | Canny magnitude | Sobel binary | Canny edges
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
axes[0].imshow(gray, cmap="gray"); axes[0].set_title("original"); axes[0].axis("off")
axes[1].imshow(sobel_mag, cmap="gray"); axes[1].set_title("Sobel magnitude"); axes[1].axis("off")
axes[2].imshow(mag, cmap="gray"); axes[2].set_title("Canny magnitude"); axes[2].axis("off")
axes[3].imshow(sobel_binary, cmap="gray"); axes[3].set_title("Sobel binary (>= 0.15)"); axes[3].axis("off")
axes[4].imshow(canny_edges, cmap="gray"); axes[4].set_title("Canny edges"); axes[4].axis("off")
plt.tight_layout()
out_path = FIG / "ex3_canny_vs_sobel.png"
fig.savefig(out_path, dpi=120)
plt.close(fig)
print("saved:", out_path)

# Summary of the differences
print()
print(f"Sobel binary edge pixels   : {int(sobel_binary.sum())} ({100*sobel_binary.mean():.2f}% of image)")
print(f"Canny edge pixels          : {int(canny_edges.sum())} ({100*canny_edges.mean():.2f}% of image)")
print()
print("Sobel mean magnitude       :", round(float(sobel_mag.mean()), 4))
print("Canny mean magnitude       :", round(float(mag.mean()), 4))


saved: /Users/danielagonzalez/Downloads/milk10k-eda/reports/figures_hw4/ex3_canny_vs_sobel.png

Sobel binary edge pixels   : 25908 (9.60% of image)
Canny edge pixels          : 21433 (7.94% of image)

Sobel mean magnitude       : 0.0705
Canny mean magnitude       : 0.1091


### Exercise 3 — Written answer

**Canny in five steps.** The `canny_from_scratch` function above
implements Canny from scratch, without using `cv2.Canny`. The five
steps are:

1. **Gaussian smoothing** (`sigma=1.4`) to reduce noise.
2. **Gradient computation** with Sobel kernels (`gx`, `gy`).
3. **Magnitude and direction** of the gradient at each pixel.
4. **Non-maximum suppression:** keep only pixels that are local maxima
   along the gradient direction, which thins thick edges to 1 pixel.
5. **Double threshold and hysteresis:** pixels above the high threshold
   are strong edges; weak pixels (between low and high) are kept only
   if they are connected to a strong edge.

**Comparison with Sobel (on `ISIC_0051817`, malignant).**

| metric | Sobel (binary at 0.15) | Canny |
|---|---|---|
| edge pixels | 25,908 (9.60% of image) | 21,433 (7.94% of image) |
| mean gradient magnitude | 0.0705 | 0.1091 |
| edge thickness | several pixels wide | one pixel wide |
| connectivity | disconnected speckle | connected curves |
| noise sensitivity | high | low (thanks to sigma=1.4 pre-blur) |
| output type | gradient magnitude + threshold | binary edge map |

**Main differences I observe.** Two things stand out from the figure
and the numbers:

- **Canny produces fewer edge pixels than a thresholded Sobel, but
  the surviving pixels have higher average gradient magnitude
  (0.109 vs 0.071).** That is exactly the effect of non-maximum
  suppression and hysteresis: a lot of medium-strength pixels are
  dropped, leaving only the strongest, locally coherent ones.
- **Canny edges form connected curves, Sobel edges are speckle.**
  Look at the lesion boundary: in the Sobel binary map it is a thick,
  noisy band; in the Canny map it is a thin, continuous outline. This
  is what makes Canny the standard choice in classical pipelines.

**Practical consequence.** For MILK10k, Canny is a much more useful
preprocessing step than a raw Sobel threshold. It gives a compact
binary map that highlights the lesion border and the main texture
lines (hairs, network) without a manually tuned threshold per image.
This is exactly what the first layers of a CNN learn to do on their
own from data, but here we have chosen the parameters by hand.
